# Feature Engineering v2 — Final Official Test

This notebook loads the fixed 50K CNN-feature + SVM artifact and evaluates it once on the official 10,000-image CIFAR-10 test set. It does not fit preprocessing, tune parameters, or train a classifier.

## 1. Imports

In [ ]:
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 30092026
classes_name = ["Airplane", "Automobile", "Bird", "Cat", "Deer", "Dog", "Frog", "Horse", "Ship", "Truck"]

## 2. Load official test features

In [ ]:
COMBINED_PATH = "combined_features_v2.npz"
MODEL_PATH = "cifar10_feature_engineering_svm_final_50k_v2.joblib"
for path in (COMBINED_PATH, MODEL_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing artifact: {path}")
with np.load(COMBINED_PATH, allow_pickle=False) as combined:
    X_test = combined["X_test_combined"].astype(np.float32)
    y_test = combined["y_test"].astype(np.int64)
    feature_order = combined["feature_order"].tolist()
    feature_dims = combined["feature_dims"].tolist()
    source_archives = combined["source_archives"].tolist()
    run_id = str(combined["run_id"].item())
    assert int(combined["seed"].item()) == SEED
assert X_test.shape == (10000, 1536) and y_test.shape == (10000,)
assert np.isfinite(X_test).all()
print("Official test rows:", len(y_test))
print("Feature sources:", source_archives)

## 3. Load and validate the frozen final artifact

In [ ]:
artifact = joblib.load(MODEL_PATH)
assert artifact["version"] == "v2"
assert artifact["seed"] == SEED
assert artifact["final_training_samples"] == 50000
assert artifact["validation_accuracy"] >= artifact["validation_gate"]
assert artifact["combined_run_id"] == run_id, "Combined archive changed after the SVM was fitted."
assert artifact["feature_order"] == feature_order
assert artifact["feature_dims"] == feature_dims
assert artifact["source_archives"] == source_archives
print("Validation selection accuracy:", f"{artifact['validation_accuracy']:.4%}")
print("Chosen weights/C/gamma:", artifact["block_weights"], artifact["C"], artifact["gamma"])

## 4. Apply the fitted training preprocessing and predict

The constant-channel mask and scaler come from the final 50K development fit. Testing never calls `.fit()`.

In [ ]:
selector = artifact["selector"]
scaler = artifact["scaler"]
svm = artifact["svm"]
block_ids = np.repeat(np.arange(3), feature_dims)[selector.get_support()]
column_weights = np.asarray(artifact["block_weights"], dtype=np.float32)[block_ids]
X_test_scaled = scaler.transform(selector.transform(X_test)).astype(np.float32)
X_test_weighted = np.ascontiguousarray(X_test_scaled * column_weights)
y_pred = svm.predict(X_test_weighted)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Final RGB + AVG + NTSC test accuracy: {test_accuracy:.4%}")
print("Target >95.00%:", "PASS" if test_accuracy > 0.95 else "NOT YET")

## 5. Classification report

In [ ]:
print(classification_report(y_test, y_pred, target_names=classes_name, digits=4))

## 6. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_name)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=45, values_format="d")
plt.title("Feature Engineering v2 — Official Test Confusion Matrix")
plt.tight_layout()
plt.show()

## 7. Per-class recall

In [ ]:
class_accuracy = np.diag(cm) / cm.sum(axis=1)
for class_name, score in zip(classes_name, class_accuracy):
    print(f"{class_name:10s}: {score:.2%}")

## 8. Final summary

In [ ]:
print("=" * 68)
print("FINAL FEATURE ENGINEERING v2 RESULT — OFFICIAL CIFAR-10 TEST")
print("=" * 68)
print("Representations:", feature_order)
print("Final SVM training rows:", artifact["final_training_samples"])
print("Validation accuracy:", f"{artifact['validation_accuracy']:.4%}")
print("Official test accuracy:", f"{test_accuracy:.4%}")
print("Target >95.00%:", "PASS" if test_accuracy > 0.95 else "NOT YET")

# Final test complete

Report the displayed accuracy as measured. Do not select a new configuration using this official test result.